In [0]:
raw_posts_df = spark.table('stackexchange_dataset.default.raw_posts')

In [0]:
from pyspark.sql.types import StructType, StructField, MapType, IntegerType, StringType
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def split_tag_to_array(df: DataFrame) -> DataFrame:
    return(
        df.withColumn('TagsArray', F.filter(F.split(F.col('Tags'), r'\|'), lambda x: x != ''))
            .drop('Tags')
    )

def rename_columns(df: DataFrame) -> DataFrame:
    return(
        df.withColumnRenamed('Id', 'PostId')
    )

def map_post_types(df: DataFrame) -> DataFrame:
    map_data = [
        (1, "Question"),
        (2, "Answer"),
        (3, "Orphaned tag wiki"),
        (4, "Tag wiki excerpt"),
        (5, "Tag wiki"),
        (6, "Moderator nomination"),
        (7, "Wiki placeholder"),
        (8, "Privilege wiki"),
        (9, "Article"),
        (10, "HelpArticle"),
        (12, "Collection"),
        (13, "ModeratorQuestionnaireResponse"),
        (14, "Announcement"),
        (15, "CollectiveDiscussion"),
        (17, "CollectiveCollection")
    ]
    map_schema = StructType([
        StructField('PostTypeId', IntegerType(), False),
        StructField('PostType', StringType(), False)
    ])
    map_df = spark.createDataFrame(map_data, map_schema)
    return(
        df.join(
            F.broadcast(map_df),
            df['PostId'] == map_df['PostTypeId'],
            'left'
        ).drop(map_df['PostTypeId'])
    )

stg_posts_df = (
    raw_posts_df
        .transform(split_tag_to_array)
        .transform(rename_columns)
        .transform(map_post_types)
) 



In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

def incremental_upsert(dest_table: str, df: DataFrame, unique_key: str, updated_at: str, full_refresh = False):
    if not spark.catalog.tableExists(dest_table) or full_refresh:
        (
            df
              .write
              .mode('overwrite')
              .option('overwriteSchema', 'true')
              .saveAsTable(dest_table)
        )
    else:
        latest_max = (
            spark.table(dest_table)
                .agg(F.max(updated_at).alias('max_ts'))
                .collect()[0]['max_ts']
        )

        incremental_df = (
            df.filter(F.col(updated_at) > latest_max)
        )

        
        delta_table = DeltaTable.forName(spark, dest_table)
        (
            delta_table.alias('t')
                .merge(
                    source = incremental_df.alias('s'),
                    condition = f't.{unique_key} = s.{unique_key}'
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
        )

dest_table = 'stackexchange_dataset.default.stg_posts'
incremental_upsert(dest_table, stg_posts_df, 'PostId', 'CreationDate')
